In [ ]:
from PIL import Image
import numpy as np
import einops
import matplotlib.pyplot as plt
import cv2
import torch

In [ ]:
def dilate_ims(ims, bbox_spec, box_h: int = 48, box_w: int = 48):

    x, y, _, _ = bbox_spec
    is_edge = 0

    if y - (box_h // 2) < 0:
        start_y = torch.tensor(0)
        is_edge = 1
    elif y + (box_h // 2) > ims[0].shape[-2]:
        start_y = torch.tensor(ims[0].shape[-2] - box_h)
        is_edge = 1
    else:
        start_y = y - (box_h // 2)

    if x - (box_w // 2) < 0:
        start_x = torch.tensor(0)
        is_edge = 1
    elif x + (box_w // 2) > ims[0].shape[-1]:
        start_x = torch.tensor(ims[0].shape[-1] - box_w)
        is_edge = 1
    else:
        start_x = x - (box_w // 2)

    specs = torch.stack([start_x, start_y, torch.tensor(is_edge)])

    x0_r, y0_r = int(start_x.round()), int(start_y.round())
    cropped_ims = [im[..., y0_r:y0_r + box_h, x0_r:x0_r + box_w] for im in ims]

    return cropped_ims, specs

In [ ]:
annot = Image.open("/home/chengjia/Downloads/nio_ucsf_12175.1_Armin_V1.tif")
seg = Image.open("/home/chengjia/Downloads/NIO_UCSF_12175_1_seg.png")

In [ ]:
annot_with_bars = np.pad(np.array(annot), ((50, 50), (50, 0),(0,0)))[:,:-50,...]

In [ ]:
annot_tiles = einops.rearrange(annot_with_bars,
                               "(h hp) (w wp) c -> h w hp wp c",
                               hp=1000,
                               wp=900)[:,:,:-100,:,:]

seg_tiles = einops.rearrange(np.array(seg),
                             "(h hp) (w wp) c -> h  w hp wp c",
                             hp=900,
                             wp=900)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(50, 100))
ax[0].imshow(annot_tiles[0][0])
ax[1].imshow(seg_tiles[0][0])

In [ ]:
for ats, sts in zip(annot_tiles, seg_tiles):
    for at, st in zip(ats, sts):
        annot_hsv = cv2.cvtColor(at, cv2.COLOR_RGB2HSV)
        annot_green = ((40 < annot_hsv[..., 0]) & (annot_hsv[..., 0] < 80)).astype(np.uint8)
        (_, _, _, annot_centroids) = cv2.connectedComponentsWithStats(annot_green)

        if not (len(annot_centroids)-1): continue

        mock_bbox = torch.tensor(
            np.hstack(
                (annot_centroids[1:], np.zeros_like(annot_centroids[1:])))).to(int)

        cropped_ims = [
            dilate_ims([
                torch.tensor(einops.rearrange(at, "h w c -> c h w")),
                torch.tensor(einops.rearrange(st, "h w c -> c h w"))
            ],
                    mb,
                    box_h=48,
                    box_w=48)[0] for mb in mock_bbox
        ]
        cropped_ims = einops.rearrange(
            torch.stack([torch.stack(ci) for ci in cropped_ims]),
            "ann v c h w -> ann v h w c"
            ).numpy()

        for ci in cropped_ims:
            fig, ax = plt.subplots(1, 2, figsize=(4, 8))
            ax[0].imshow(ci[0])
            ax[1].imshow(ci[1])

In [ ]:
cropped_ims = [dilate_ims([
    torch.tensor(einops.rearrange(annot_tiles[0][0], "h w c -> c h w")),
    torch.tensor(einops.rearrange(seg_tiles[0][0], "h w c -> c h w"))
],
                             mb,
                             box_h=48,
                             box_w=48)[0] for mb in mock_bbox]

In [ ]:
cropped_ims = einops.rearrange(torch.stack([torch.stack(ci) for ci in cropped_ims]), "ann v c h w -> ann v h w c").numpy()

In [ ]:
cropped_ims.shape

In [ ]:
for ci in cropped_ims:
    fig, ax = plt.subplots(1, 2, figsize=(4, 8))
    ax[0].imshow(ci[0])
    ax[1].imshow(ci[1])